In [32]:
import openpyxl
import os
from collections import Counter
from datetime import datetime, timedelta

CALL_CODES = {'B03.044.001.1', 'B03.044.001.2'}
START = datetime(2025, 1, 1)
END = datetime(2026, 9, 30)

In [33]:
def read_files(folder):
    
    all_days = []
    current = START
    while current <= END:
        all_days.append(current)
        current += timedelta(days=1)

    all_months = []
    current_key = (START.year, START.month)
    end_key = (END.year, END.month)
    while current_key <= end_key:
        all_months.append(current_key)
        year, month = current_key
        if month == 12:
            current_key = (year + 1, 1)
        else:
            current_key = (year, month + 1)

    calls_per_day = {day: 0 for day in all_days}
    calls_per_month = {m: 0 for m in all_months}
    revenue_per_month = {m: 0.0 for m in all_months}

    service_counts = Counter()
    service_revenue = {}
    diagnosis_counts = Counter()

    files = os.listdir(folder)
    for file in files:
        path = os.path.join(folder, file)
        wb = openpyxl.load_workbook(path)
        ws = wb.active
        for row in ws.iter_rows(min_row=2, values_only=True):
            
            if row[0] == 'Итого по реестру' or row[0] is None:
                continue

            date = row[1]
            code = row[2]
            quantity = row[5] 
            total = row[6] 
            diagnosis = row[7]

            key = (date.year, date.month)

            if code in CALL_CODES:
                calls_per_day[date] = calls_per_day.get(date, 0) + quantity
                calls_per_month[key] = calls_per_month.get(key, 0) + quantity

                if diagnosis is not None:
                    diagnosis_group = str(diagnosis)[:3]
                    diagnosis_counts[diagnosis_group] += 1

            revenue_per_month[key] = revenue_per_month.get(key, 0.0) + total
            service_counts[code] += quantity
            service_revenue[code] = service_revenue.get(code, 0.0) + total

    months_order = sorted(calls_per_month.keys())

    return {
        'calls_per_day': calls_per_day,
        'calls_per_month': calls_per_month,
        'revenue_per_month': revenue_per_month,
        'service_counts': service_counts,
        'service_revenue': service_revenue,
        'diagnosis_counts': diagnosis_counts,
        'months_order': months_order,
    }
            

In [36]:
def mean(values):
    return sum(values) / len(values)

def variance(values):
    m = mean(values)
    return sum((x - m) ** 2 for x in values) / len(values)

def std(values):
    return variance(values) ** 0.5

def median(values):
    s = sorted(values)
    n = len(s)
    if n % 2 == 1:
        return s[n // 2]
    else:
        return (s[n // 2 - 1] + s[n // 2]) / 2

def mode(values):
    return Counter(values).most_common(1)[0][0]

def minimum(values):
    return min(values)

def maximum(values):
    return max(values)

def covariance(xs, ys):
    mx = mean(xs)
    my = mean(ys)
    n = len(xs)
    return sum((x - mx) * (y - my) for x, y in zip(xs, ys)) / n

def correlation(xs, ys):
    return covariance(xs, ys) / (std(xs) * std(ys))

def describe(values, name="данные"):
    print(f"--- {name} ---")
    print(f"{'Математическое ожидание':<25} {mean(values):>15.2f}")
    print(f"{'Дисперсия':<25} {variance(values):>15.2f}")
    print(f"{'Стандартное отклонение':<25} {std(values):>15.2f}")
    print(f"{'Медиана':<25} {median(values):>15.2f}")
    print(f"{'Мода':<25} {mode(values):>15}")
    print(f"{'Мин':<25} {minimum(values):>15}")
    print(f"{'Макс':<25} {maximum(values):>15}")
    print()

def describe_categorical(counter, name="данные", top_n=10):
    total = sum(counter.values())
    print(f"--- {name} ---")
    print(f"Всего: {total}")
    for value, count in counter.most_common(top_n):
        share = count / total * 100
        print(f"{value}: {count} ({share:.2f}%)")
    print()

In [ ]:
data = read_files('data')

calls_day = list(data['calls_per_day'].values())
describe(calls_day, "Число вызовов в день")

calls_month = list(data['calls_per_month'].values())
describe(calls_month, "Число вызовов в месяц")

revenue_month = list(data['revenue_per_month'].values())
describe(revenue_month, "Выручка за месяц")

describe_categorical(data['service_counts'], "Коды услуг")
describe_categorical(data['diagnosis_counts'], "Диагнозы")

months = data['months_order']
calls = [data['calls_per_month'][m] for m in months]
revenue = [data['revenue_per_month'][m] for m in months]

month_numbers = list(range(1, len(months) + 1))

print(f"Корреляция (вызовы vs месяц):  {correlation(month_numbers, calls):.3f}")
print(f"Корреляция (выручка vs месяц): {correlation(month_numbers, revenue):.3f}")

--- Число вызовов в день ---
Математическое ожидание              1.80
Дисперсия                            2.16
Стандартное отклонение               1.47
Медиана                              2.00
Мода                                    1
Мин                                     0
Макс                                    9

--- Число вызовов в месяц ---
Математическое ожидание             54.76
Дисперсия                          345.51
Стандартное отклонение              18.59
Медиана                             50.00
Мода                                   39
Мин                                    18
Макс                                  108

--- Выручка за месяц ---
Математическое ожидание         493238.10
Дисперсия                  34830848072.56
Стандартное отклонение          186630.24
Медиана                         449000.00
Мода                             326500.0
Мин                              144000.0
Макс                             925000.0

--- Коды услуг ---
Всего: 1425.